# Advanced Workflows

Incremental updates, caching, and production patterns.

In [ ]:
from vulnflow import VulnDataset
from vulnflow.transforms.flatten import CVSSFlattenTransform, CWEPrimaryTransform, CPEPrimaryTransform
from vulnflow_data import download_modified, build_cache
import tempfile
from pathlib import Path
import numpy as np

## Download Modified feed

Last 8 days of CVE changes (~2-5K records).

In [ ]:
modified_path = download_modified()
modified_path

## Load and flatten

In [ ]:
ds = VulnDataset.from_nvd(modified_path).transform([
    CVSSFlattenTransform(),
    CWEPrimaryTransform(),
    CPEPrimaryTransform(),
])

df = ds.to_pandas()
len(df)

## Incremental pattern: Cache + Delta

Build once, merge every session.

In [ ]:
# Setup: Create cache and delta (simulated)
demo_dir = Path(tempfile.mkdtemp())
cache_path = demo_dir / "cache.parquet"
delta_path = demo_dir / "delta.parquet"

# Cache: First 1000 CVEs
cache_ds = VulnDataset.from_nvd(modified_path).transform([
    CVSSFlattenTransform(),
    CWEPrimaryTransform(),
    CPEPrimaryTransform(),
])

# Limit to first 1000 for demo
cache_df = cache_ds.to_pandas().head(1000)
cache_limited = VulnDataset()
cache_limited._data = {col: cache_df[col].values for col in cache_df.columns}
cache_limited.to_parquet(cache_path)

print(f"Cache: {len(cache_limited)} CVEs")

In [ ]:
# Delta: Simulate update (1 modified + 1 new)
cache_sample_df = VulnDataset.from_parquet(cache_path).to_pandas()
sample_cve_id = cache_sample_df.iloc[0]['cve_id']

delta_ds = VulnDataset()
delta_ds._data = {
    "cve_id": np.array([sample_cve_id, "CVE-9999-NEW"], dtype=object),
    "description": np.array(["UPDATED: Score changed to 10.0", "NEW: Just disclosed"], dtype=object),
    "cvss_base_score": np.array([10.0, 7.5], dtype=object),
    "cvss_base_severity": np.array(["CRITICAL", "HIGH"], dtype=object),
}
delta_ds.to_parquet(delta_path)

print(f"Delta: {len(delta_ds)} CVEs (1 update, 1 new)")

## Merge with upsert

Delta automatically overwrites cache.

In [ ]:
# Merge cache + delta
merged_df = VulnDataset.from_parquet([str(cache_path), str(delta_path)]).to_pandas()

print(f"Merged: {len(merged_df)} CVEs")
print(f"Expected: 1000 (cache) + 1 (new) = 1001")

## Verify upsert

In [ ]:
# Check updated CVE
updated = merged_df[merged_df['cve_id'] == sample_cve_id].iloc[0]
updated[['cve_id', 'cvss_base_score', 'description']]

In [ ]:
# Check new CVE
new = merged_df[merged_df['cve_id'] == 'CVE-9999-NEW'].iloc[0]
new[['cve_id', 'cvss_base_score', 'description']]

## Production pattern

```python
from vulnflow_data import download_recent, download_modified, build_cache

# ONE-TIME: Build cache from Recent feed (~240K CVEs)
recent_path = download_recent()
cache_path = build_cache([recent_path], output_path="data/nvd_cache.parquet")

# EVERY SESSION: Download Modified, merge with cache
modified_path = download_modified()
delta_cache = build_cache([modified_path], output_path="data/nvd_delta.parquet")

# Load merged data (~0.2s for 240K + 2K)
df = VulnDataset.from_parquet(["data/nvd_cache.parquet", "data/nvd_delta.parquet"]).to_pandas()
```

In [ ]:
# Cleanup
import shutil
shutil.rmtree(demo_dir)